# Destek-direnc araliginda al-sat | gecmis veride deneme

## Sistem

* **Tepe cizgisi (direnc):** son 1 haftanin en yuksek fiyati
* **Dip cizgisi (destek):** son 1 haftanin en dusuk fiyati
* **Al:** fiyat dibe yaklasinca - dipten, kanal genisliginin %10'u kadar yukarisina inerse
* **Sat:** fiyat tepe cizgisine degince, tam cizgide
* **Zararina sat (stop):** fiyat, alirken beklenen dibin cok altina inerse -
  dipten, kanal genisliginin yarisi kadar asagi

Ornek: tepe 510, dip 500 (genislik 10)

| | seviye |
|---|---|
| al | 501 |
| sat | 510 (cizgi her bar guncellenir) |
| zararina sat | 495 (alista sabitlenir) |

Cizgiler **sadece onceki barlardan** cizilir, o anki bar dahil edilmez.
Bar icinde neyin once oldugu bilinmediginde hep **kotu** olan secilir
(ayni barda hem stop hem tepe gorulduyse once stop sayilir).
Ayrintilar: `lab/backtest.py` basindaki not.

## Deney kaydi (2026-09-11)

**Sistem:** fiyat son N gunun en dusuk seviyesine yaklasinca al; en yuksek
seviyesine degince ya da en dusugun epey altina inince sat.

**Baseline (karsilastirma):** 2016 basinda al, hicbir sey yapma, 2024
Haziran'da sat.

**Secilen esikler:**

* al: dipten, iki cizgi arasindaki mesafenin %10'u yukarisi
* zararina sat: dipten, iki cizgi arasindaki mesafenin yarisi asagisi (alista sabitlenir)
* maliyet: her alista ve her satista %0.05

**Veri:** SPY, 2016-01 -> 2024-06. 2024-07 -> 2026-09 arasi saklanan veri
**acilmadi**.

### Sonuclar - 1 lira ne oldu (maliyet dahil)

**Tum donem, 1 haftalik cizgiler (2016 -> 2024-06)**

| | 1 lira -> |
|---|---|
| gunluk veri | 1.03 |
| saatlik veri | 1.11 |
| baseline | 3.12 |

Isleme giren esikler degistirildi (12 ayar): en iyisi 1.57, yine baseline'in
cok altinda. Zararina satis kaldirilinca sonuc iyilesiyor.

**Egitim / test (2019 -> 2024-06):** her yil esikler sadece onceki yillara
bakarak secildi, o yilda denendi.

| | 1 lira -> |
|---|---|
| gunluk veri | 1.09 |
| saatlik veri | 1.22 |
| baseline | 2.37 |

6 yilin hicbirinde baseline gecilmedi. 2022 dusus yilinda sistem de ~%18
kaybetti. Egitim her seferinde en uzak zararina satis seviyesini secti.

**Daha kisa cizgiler, saatlik veri (bolum 6)**

| cizgi | islem | tum donem | maliyet olmasa | egitim/test 2019-24 | baseline'i gectigi yil |
|---|---|---|---|---|---|
| 1 gun | 1230 | 0.56 | 1.91 | 1.06 | 6 yilin 1'i |
| 2 gun | 676 | 0.73 | 1.43 | 0.96 | hic |
| 3 gun | 471 | 1.16 | 1.85 | 1.16 | 6 yilin 1'i |
| 5 gun | 277 | 1.11 | 1.46 | 1.22 | hic |
| baseline | 1 | 3.12 | 3.12 | 2.37 | - |

Cizgi kisaldikca islem sayisi artiyor, maliyet kazanci yiyor. Maliyet hic
olmasa bile hicbiri baseline'i gecmiyor.

**Anormal donem mi (bolum 7):** kar rakamlari 20 gunluk bir kesitten
degil, 2016-2024 arasi 8.5 yilin tamamindan geliyor. Defterdeki grafikler
sadece son gunleri yakinlastirip gosteriyor. Yil yil bakinca 1 haftalik
gunluk sistem 9 yilin sadece 1'inde (2022) baseline'i gecti. 2018'de piyasa
%5 dustu, sistem %11.5 kaybetti. SPY bu donemde yilda ~%14 yukseldi,
uzun donem ortalamasindan (~%10) guclu. Bu, dipte alip tepede satan bir
sisteme karsi calisir, ama yatay/dusus yillarinda da sistem tutarli bir
ustunluk gostermedi.

**Uzun cizgiler, gunluk veri (bolum 7)**

| cizgi | islem | tum donem | maliyet olmasa | egitim/test 2019-24 | baseline'i gectigi yil |
|---|---|---|---|---|---|
| 5 gun | 237 | 1.03 | 1.30 | 1.09 | 6 yilin 1'i |
| 10 gun | 121 | 1.14 | 1.29 | 1.15 | hic |
| 20 gun | 61 | 1.21 | 1.29 | 1.06 | hic |
| 50 gun | 21 | 1.20 | 1.22 | 1.20 | 6 yilin 1'i |
| 100 gun | 8 | 1.13 | 1.14 | 1.23 | 6 yilin 1'i |
| baseline | 1 | 3.12 | 3.12 | 2.37 | - |

Uzun cizgide islem azaliyor, maliyet daha az yiyor, sonuc biraz iyilesiyor.
Ama sistem zamanin ~%75'inde nakitte bekliyor ve islemlerin yarisi yine
zararina satisla kapaniyor. 100 gunde 8.5 yilda sadece 8 islem var; bu kadar
az islemle sonuca guvenmek zor. 50 gunluk cizgi 2022'de neredeyse hic
kaybetmedi (-%0.7, piyasa -%18), ama 2018'de piyasadan kotu (-%11.7, piyasa -%5).

**Bollinger + takip eden stop, SPY ve AAPL (bolum 8)**

Yeni kurallar (kullanicinin secimi):

* cizgi: son 20 gunun ortalama kapanisi +/- 2 sapma (Bollinger). Fiyat
  yukselince bant da yukselir.
* satis: tepeye degince satma. O andan sonraki en yuksek fiyattan, tepeye
  degildigi andaki genisligin `takip_payi` kati kadar duserse sat. Seviye
  sadece yukari gider. Varsayilan `takip_payi` = 0.5 (benim secimim).

Egitim/test 2019 -> 2024-06, surum ve esikler her yil sadece egitimden
secildi (1 lira ->):

| | gunluk | saatlik | baseline |
|---|---|---|---|
| SPY | 1.78 | 1.81 | 2.37 |
| AAPL | 5.80 | 6.29 | 5.63 |

Takip eden stop sonucu buyuk olcude iyilestirdi (SPY 1.06 -> 1.78).
AAPL'da baseline'in biraz ustune cikti.

**Ama iyilesmenin kaynagi:** egitim her yil en genis takip mesafesini (1.0)
secti. Takip mesafesi genisledikce sistem neredeyse hic satmiyor:

| takip mesafesi | SPY piyasada | SPY 1 lira | AAPL piyasada | AAPL 1 lira |
|---|---|---|---|---|
| 0.5 | %67 | 1.95 | %62 | 3.49 |
| 1.0 | %90 | 2.56 | %90 | 10.44 |
| 2.0 | %98 | 3.37 | %98 | 11.22 |
| baseline | %100 | 3.12 | %100 | 8.95 |

(gunluk, eski cizgi, 2016 -> 2024-06, tum donem)

Yani sistem iyilestikce **baseline'a donusuyor**: zamanin %90-98'inde
piyasada, en buyuk erime baseline ile ayni (~%35-39). Baseline'i gectigi
durumlar 8.5 yilda 6-10 islemle oluyor; bu kadar az islemle farkin sans
olmadigini soylemek mumkun degil. Dususte koruma yok.

**Karar:** sistem bu haliyle baseline'i gecemedi.

**Bolum 8 sonrasi:** takip eden stop sistemi baseline'a yaklastirdi, ama bunu piyasada neredeyse surekli kalarak yapiyor. Baseline'dan ayri bir ustunluk (dususte koruma ya da ayni riskle daha fazla kazanc) henuz gosterilmedi.

**KABUL-1 (2026-09-11, kullanici karari, simdilik):** AAPL, eski cizgi
(son 20 gunun en yuksegi/en dusugu) + takip eden stop. Alim 0.10, stop 2.0,
takip mesafesi 1.0 (egitimin her yil sectigi). Egitim/test 2019 -> 2024-06:
gunluk 5.80, saatlik 6.29, baseline 5.63 / 5.61. Sinirlari ve dogrulama
adimlari: `hypotheses/REGISTRY.md` KABUL-1.

## 0. Ayarlar

In [ ]:
SYMBOL    = 'SPY'
HAFTA_GUN = 5          # cizgiler 1 haftalik
ALIM_PAYI = 0.10       # dipten kanal genisliginin %10'u yukarisi -> al
STOP_PAYI = 0.50       # dipten kanal genisliginin yarisi asagisi -> zararina sat
MALIYET   = 0.0005     # her alista ve her satista %0.05

## 1. Veri

Sadece **2016 - 2024 Haziran** kullaniliyor. 2024 Temmuz sonrasi saklanan
kisim bu defterde acilmiyor; en son, sistem kesinlestikten sonra bir kez
test edilecek.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd

from src.data.load import load_bars
from lab.session import regular_hours
from lab.splits import split_research_vault
from lab.backtest import Kurallar, calistir, ozet
from lab import plots

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

gunluk  = split_research_vault(load_bars(SYMBOL, '1Day'), horizon_bars=1).research
saatlik = split_research_vault(regular_hours(load_bars(SYMBOL, '1Hour')), horizon_bars=1).research
print(f'gunluk : {len(gunluk):>6} bar')
print(f'saatlik: {len(saatlik):>6} bar  (sadece borsa saatleri, 09:30-16:00)')

## 2. Gunluk

Fiyata gunde bir kez bakilir. Cizgiler son 5 gunden cizilir.

In [ ]:
k_gunluk = Kurallar(n=HAFTA_GUN, alim_payi=ALIM_PAYI, stop_payi=STOP_PAYI, maliyet=MALIYET)
s_gunluk = calistir(gunluk, k_gunluk)
ozet(gunluk, s_gunluk, bar_per_yil=252)

In [ ]:
plots.trades_chart(gunluk, s_gunluk, last=120);

In [ ]:
plots.equity_chart(gunluk, s_gunluk);

## 3. Saatlik

Fiyata saatte bir kez bakilir. Cizgiler yine son 1 haftadan cizilir
(5 gun x 7 saat = 35 bar). Ani dususte gun sonunu beklemeden satabilir.

Not: gece borsa kapaliyken olan dususu saatlik de yakalayamaz; sabah
acilistan satar.

In [ ]:
k_saatlik = Kurallar(n=HAFTA_GUN * 7, alim_payi=ALIM_PAYI, stop_payi=STOP_PAYI, maliyet=MALIYET)
s_saatlik = calistir(saatlik, k_saatlik)
ozet(saatlik, s_saatlik, bar_per_yil=252 * 7)

In [ ]:
plots.trades_chart(saatlik, s_saatlik, last=7 * 20);

In [ ]:
plots.equity_chart(saatlik, s_saatlik);

## 4. Esikler degisince sonuc degisiyor mu

Yukaridaki %10 ve yarim genislik esiklerini ben sectim. Sonucun bu secime
bagli olup olmadigini gormek icin birkac farkli degerle tekrar calistiriyoruz.

**Buradaki en iyi sonucu secip "sistem bu" demek hile olur:** 12 deneme
yapip en iyisini secmek, sans eseri iyi cikani secmektir. Bu tablo sadece
"secim sonucu degistiriyor mu" sorusu icin.

In [ ]:
rows = []
for veri, n in ((gunluk, HAFTA_GUN), (saatlik, HAFTA_GUN * 7)):
    ad = 'gunluk' if veri is gunluk else 'saatlik'
    for a in (0.05, 0.10, 0.20):
        for sp in (0.25, 0.50, 1.00, None):
            k = Kurallar(n=n, alim_payi=a, stop_payi=sp if sp else 1e9, maliyet=MALIYET)
            rows.append({'veri': ad, 'alim_payi': a,
                         'stop_payi': sp if sp else 'stop yok',
                         'toplam_getiri': calistir(veri, k).bakiye.iloc[-1] - 1})
esikler = pd.DataFrame(rows).pivot_table(index=['veri', 'alim_payi'],
                                         columns='stop_payi', values='toplam_getiri',
                                         aggfunc='first')
esikler

## 5. Egitim ve test

Esikleri gecmis yillara bakarak sec, hic bakilmamis bir sonraki yilda dene:

* **Egitim:** test yilindan ONCEKI tum veride 12 esik cifti denenir, en cok
  kazandiran secilir.
* **Test:** secilen esikler o yilda uygulanir. Esik secerken bu yila hic
  bakilmadi.

Bu 2019'dan 2024 Haziran'a kadar her yil icin tekrarlanir.

`egitimde` ile `testte` arasindaki fark onemli: egitimde gorunen kazanc,
esik o veriye bakilarak secildigi icin hep abartilidir. Gercek beklenti
`testte` sutunudur.

In [ ]:
from lab.backtest import egitim_test

def zincir(t):
    # yillik sonuclari arka arkaya bagla: 1 lira 2019 basindan 2024 Haziran'a ne oldu
    return t[['testte', 'varsayilan_esiklerle', 'basta_al_sonda_sat']].add(1).prod()

et_gunluk  = egitim_test(gunluk,  n=HAFTA_GUN,     test_yillari=range(2019, 2025), maliyet=MALIYET)
et_saatlik = egitim_test(saatlik, n=HAFTA_GUN * 7, test_yillari=range(2019, 2025), maliyet=MALIYET)
et_gunluk

In [ ]:
et_saatlik

In [ ]:
pd.DataFrame({'gunluk': zincir(et_gunluk), 'saatlik': zincir(et_saatlik)})

## 6. Daha kisa cizgiler (saatlik)

Cizgiler 1 hafta yerine daha kisa sureden cizilir:

| cizgi | saatlik bar |
|---|---|
| 1 gun | 7 (son 1 islem gununun en yuksegi / en dusugu) |
| 2 gun | 14 |
| 3 gun | 21 |
| 5 gun | 35 (yukaridaki 1 hafta) |

Kurallar ve esikler ayni. Her biri icin hem tum donem hem egitim/test
calistirilir.

In [ ]:
rows, sonuclar = [], {}
for gun in (1, 2, 3, 5):
    k = Kurallar(n=gun * 7, alim_payi=ALIM_PAYI, stop_payi=STOP_PAYI, maliyet=MALIYET)
    s = calistir(saatlik, k)
    o = ozet(saatlik, s, bar_per_yil=252 * 7)
    et = egitim_test(saatlik, n=gun * 7, test_yillari=range(2019, 2025), maliyet=MALIYET)
    sonuclar[gun] = s
    rows.append({
        'cizgi (gun)': gun,
        'islem': len(s.islemler),
        'stopla kapanan': (s.islemler['sebep'] == 'stop').mean(),
        'tum donem': 1 + o.loc['toplam getiri', 'sistem'],
        'maliyet olmasa': 1 + o.loc['toplam getiri (maliyetsiz)', 'sistem'],
        'baseline': 1 + o.loc['toplam getiri', 'başta al, sonda sat'],
        'egitim/test': et['testte'].add(1).prod(),
        'egitim/test baseline': et['basta_al_sonda_sat'].add(1).prod(),
        'baseline gectigi yil': int((et['testte'] > et['basta_al_sonda_sat']).sum()),
    })
kisa = pd.DataFrame(rows).set_index('cizgi (gun)')
kisa

1 gunluk cizgilerle son 10 gunun al-sat noktalari:

In [ ]:
plots.trades_chart(saatlik, sonuclar[1], last=7 * 10);

## 7. Anormal donem mi + daha uzun cizgiler (gunluk)

**Anormal donem kontrolu:** sistemin getirisi yil yil baseline ile yan yana.
Piyasanin yatay ya da dususte oldugu yillarda (2018, 2022) sistem daha iyi
mi?

**Uzun cizgiler:** gunluk barda cizgiler 5, 10, 20, 50 ve 100 gunden
cizilir (1 hafta, 2 hafta, 1 ay, 2.5 ay, 5 ay). Kurallar ve esikler ayni.

In [ ]:
def yil_yil(veri, sonuc):
    # her takvim yilinda sistem ve baseline ne kazandirdi
    yil = veri['timestamp'].dt.year.to_numpy()
    eq, c = sonuc.bakiye.to_numpy(), veri['close'].to_numpy()
    rows = []
    for y in sorted(set(yil)):
        idx = [i for i in range(len(yil)) if yil[i] == y]
        o = idx[0] - 1
        rows.append({'yil': y,
                     'sistem': eq[idx[-1]] / (eq[o] if o >= 0 else 1.0) - 1,
                     'baseline': c[idx[-1]] / c[max(o, 0)] - 1})
    return pd.DataFrame(rows).set_index('yil')

rows, uzun = [], {}
for gun in (5, 10, 20, 50, 100):
    k = Kurallar(n=gun, alim_payi=ALIM_PAYI, stop_payi=STOP_PAYI, maliyet=MALIYET)
    s = calistir(gunluk, k)
    o = ozet(gunluk, s, bar_per_yil=252)
    et = egitim_test(gunluk, n=gun, test_yillari=range(2019, 2025), maliyet=MALIYET)
    uzun[gun] = s
    rows.append({
        'cizgi (gun)': gun,
        'islem': len(s.islemler),
        'ort. tutma (gun)': s.islemler['bar'].mean(),
        'piyasada': o.loc['piyasada kalma süresi', 'sistem'],
        'stopla kapanan': (s.islemler['sebep'] == 'stop').mean(),
        'tum donem': 1 + o.loc['toplam getiri', 'sistem'],
        'maliyet olmasa': 1 + o.loc['toplam getiri (maliyetsiz)', 'sistem'],
        'en buyuk erime': o.loc['en büyük düşüş', 'sistem'],
        'egitim/test': et['testte'].add(1).prod(),
        'egitim/test baseline': et['basta_al_sonda_sat'].add(1).prod(),
        'baseline gectigi yil': int((et['testte'] > et['basta_al_sonda_sat']).sum()),
    })
pd.DataFrame(rows).set_index('cizgi (gun)')

In [ ]:
# yil yil: her cizgi suresi icin sistemin getirisi, en sagda baseline
yillik = pd.DataFrame({f'{g} gun': yil_yil(gunluk, s)['sistem'] for g, s in uzun.items()})
yillik['baseline'] = yil_yil(gunluk, uzun[5])['baseline']
yillik

50 gunluk cizgilerle son 1 yilin al-sat noktalari ve 1 liranin seyri:

In [ ]:
plots.trades_chart(gunluk, uzun[50], last=250);

In [ ]:
plots.equity_chart(gunluk, uzun[50]);

## 8. Bollinger + takip eden stop, SPY ve AAPL

4 surum x 2 sembol x 2 veri:

| surum | cizgi | satis |
|---|---|---|
| eski cizgi + tepede sat | son 20 gunun en yuksegi/en dusugu | tepe cizgisinde |
| eski cizgi + takip eden stop | son 20 gunun en yuksegi/en dusugu | takip eden stop |
| Bollinger + tepede sat | 20 gunluk ortalama +/- 2 sapma | tepe cizgisinde |
| Bollinger + takip eden stop | 20 gunluk ortalama +/- 2 sapma | takip eden stop |

Gunlukte 20 bar, saatlikte 140 bar (20 gun x 7). SPY bir endeks fonu,
AAPL tek bir buyuk hisse. Her tipten sadece bir ornek var; sonuc "SPY icin /
AAPL icin" diye okunmali, "tum fonlar / tum hisseler" diye degil.

Bu hucre birkac dakika surer.

In [ ]:
SURUMLER = {
    'eski cizgi + tepede sat':      {'cizgi': 'donchian',  'satis': 'tepe'},
    'eski cizgi + takip eden stop': {'cizgi': 'donchian',  'satis': 'takip'},
    'Bollinger + tepede sat':       {'cizgi': 'bollinger', 'satis': 'tepe'},
    'Bollinger + takip eden stop':  {'cizgi': 'bollinger', 'satis': 'takip'},
}
VERI = {
    ('SPY', 'gunluk'):   (gunluk, 20, 252),
    ('SPY', 'saatlik'):  (saatlik, 140, 252 * 7),
    ('AAPL', 'gunluk'):  (split_research_vault(load_bars('AAPL', '1Day'), horizon_bars=1).research, 20, 252),
    ('AAPL', 'saatlik'): (split_research_vault(regular_hours(load_bars('AAPL', '1Hour')), horizon_bars=1).research, 140, 252 * 7),
}

rows, ets, v2 = [], {}, {}
for (sembol, veri_ad), (veri, n, bpy) in VERI.items():
    for surum, ayar in SURUMLER.items():
        s = calistir(veri, Kurallar(n=n, maliyet=MALIYET, **ayar))
        o = ozet(veri, s, bar_per_yil=bpy)
        et = egitim_test(veri, n=n, test_yillari=range(2019, 2025), maliyet=MALIYET, sabit=ayar)
        v2[(sembol, veri_ad, surum)], ets[(sembol, veri_ad, surum)] = s, et
        rows.append({
            'sembol': sembol, 'veri': veri_ad, 'surum': surum,
            'islem': len(s.islemler),
            'tum donem': 1 + o.loc['toplam getiri', 'sistem'],
            'en buyuk erime': o.loc['en büyük düşüş', 'sistem'],
            'baseline': 1 + o.loc['toplam getiri', 'başta al, sonda sat'],
            'egitim/test': et['testte'].add(1).prod(),
            'egitim/test baseline': et['basta_al_sonda_sat'].add(1).prod(),
            'baseline gectigi yil': int((et['testte'] > et['basta_al_sonda_sat']).sum()),
        })
pd.DataFrame(rows).set_index(['sembol', 'veri', 'surum'])

### Davranis listesi

Her test yilinda hem **surum** hem **esikler** sadece o yildan onceki
yillara bakilarak secilir (egitimde en cok kazandiran). Sonra o yilda
denenir. Test yilina bakilarak secim yapilmaz.

In [ ]:
davranis = []
for sembol, veri_ad in VERI:
    for i, yil in enumerate(range(2019, 2025)):
        aday = {sv: ets[(sembol, veri_ad, sv)].iloc[i] for sv in SURUMLER}
        en = max(aday, key=lambda sv: aday[sv]['egitimde'])
        r = aday[en]
        davranis.append({'sembol': sembol, 'veri': veri_ad, 'yil': yil,
                         'secilen surum': en, 'alim': r['secilen_alim'],
                         'stop': r['secilen_stop'], 'takip': r.get('secilen_takip'),
                         'testte': r['testte'], 'baseline': r['basta_al_sonda_sat']})
davranis = pd.DataFrame(davranis)
davranis

In [ ]:
# 2019 basindan 2024 Haziran'a 1 lira
davranis.groupby(['sembol', 'veri'])[['testte', 'baseline']].apply(lambda t: t.add(1).prod())

### Iyilesme nereden geliyor

Takip mesafesi genisledikce sistem daha az satiyor ve daha uzun sure
piyasada kaliyor. Asagidaki tablo mesafeye gore piyasada kalma suresini
gosterir (gunluk, 2016 -> 2024-06).

In [ ]:
rows = []
for sembol in ('SPY', 'AAPL'):
    veri = VERI[(sembol, 'gunluk')][0]
    for cz in ('donchian', 'bollinger'):
        for tk in (0.5, 1.0, 2.0, 4.0):
            k = Kurallar(n=20, cizgi=cz, satis='takip', alim_payi=0.10, stop_payi=2.0,
                         takip_payi=tk, maliyet=MALIYET)
            s = calistir(veri, k)
            o = ozet(veri, s, bar_per_yil=252)
            rows.append({'sembol': sembol, 'cizgi': cz, 'takip mesafesi': tk,
                         'piyasada': o.loc['piyasada kalma süresi', 'sistem'],
                         'islem': len(s.islemler),
                         '1 lira': 1 + o.loc['toplam getiri', 'sistem'],
                         'baseline': 1 + o.loc['toplam getiri', 'başta al, sonda sat'],
                         'en buyuk erime': o.loc['en büyük düşüş', 'sistem'],
                         'baseline erime': o.loc['en büyük düşüş', 'başta al, sonda sat']})
pd.DataFrame(rows).set_index(['sembol', 'cizgi', 'takip mesafesi'])

SPY ve AAPL, Bollinger + takip eden stop (varsayilan esikler), gunluk:

In [ ]:
plots.trades_chart(VERI[('SPY', 'gunluk')][0], v2[('SPY', 'gunluk', 'Bollinger + takip eden stop')], last=250);

In [ ]:
plots.equity_chart(VERI[('SPY', 'gunluk')][0], v2[('SPY', 'gunluk', 'Bollinger + takip eden stop')]);

In [ ]:
plots.trades_chart(VERI[('AAPL', 'gunluk')][0], v2[('AAPL', 'gunluk', 'Bollinger + takip eden stop')], last=250);

In [ ]:
plots.equity_chart(VERI[('AAPL', 'gunluk')][0], v2[('AAPL', 'gunluk', 'Bollinger + takip eden stop')]);

## Nasil okunur

* **toplam getiri:** 1 lira bu donemde kac lira kazandirdi (0.10 = %10)
* **en buyuk dusus:** para bir ara zirvesinden en fazla ne kadar eridi
* **piyasada kalma suresi:** zamanin ne kadarinda elde hisse vardi
* **kazanan islem orani:** islemlerin kacinda maliyetten sonra kar edildi

Sistemin ise yaramasi icin **baslangicta al, sonda sat**'i gecmesi gerekir.
Gecemiyorsa, ayni parayi alip hic dokunmadan beklemek daha iyidir.